# IPI Read/Control Head PoC — 실험 노트북

이 노트북은 지금까지 Colab에서 진행한 실험(Phase 0 스모크 테스트 → Phase 1~3 본 실험 →
3B에서 발견된 `read_token_prob` 측정 버그 디버깅 → 수정)을 순서대로 재현할 수 있게 정리한 것입니다.

VS Code의 Colab 확장(런타임 연결)으로 이 노트북을 열고 셀을 순서대로 실행하면 됩니다.

**전제**: Colab 런타임의 하드웨어 가속기를 GPU(T4 이상)로 설정할 것.

환경/버전 노트 (RUN.md, results/2026-07-27_colab_smoketest 참고):
- Colab 기본 `transformers`는 `lxt`가 요구하는 구버전 API와 최신 아키텍처 지원 사이에서 충돌이 남
- `transformers==4.51.3`으로 고정하면 해결됨 (그보다 최신이면 `find_pruneable_heads_and_indices` 없음 에러,
  그보다 예전이면 `qwen3` 모듈 없음 에러)
- pip 설치 후 **런타임 재시작 필수** (이미 import된 구버전 transformers가 메모리에 남아있음)

## 0. 저장소 clone + 의존성 설치

In [ ]:
!git clone http://github.com/jongbin03/head_poc.git

In [ ]:
%cd head_poc

In [ ]:
!pip install -q "transformers==4.51.3" lxt accelerate bitsandbytes matplotlib
# gradio-huggingface_hub 버전 충돌 경고가 뜨지만, gradio를 쓰지 않으므로 무시 가능.

### ⚠️ 위 셀 실행 후 런타임을 반드시 재시작할 것

메뉴 `런타임 > 세션 다시 시작` 또는 아래 셀 실행. 재시작 후 `%cd head_poc`부터 다시 실행하지 않아도
되도록, 이 노트북은 재시작 이후 셀(1번부터)이 각자 필요한 경로로 다시 이동하도록 작성돼 있습니다.

In [ ]:
import os
os.kill(os.getpid(), 9)

## 1. Phase 0 — 극소형 모델(0.5B)로 파이프라인 자체 검증 (smoke test)

GPU 메모리 걱정 없이 코드 버그(shape, span 오프셋, grad None 등)를 먼저 잡는다.

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --family qwen2 \
  --device cuda \
  --topk 20 \
  --dataset_limit 2

**체크리스트** (RUN.md 3단계):
1. `[1/4]`~`[4/4]` 네 단계가 에러 없이 순서대로 출력
2. `top-20 Jaccard(...)` 세 줄이 0~1 사이 (NaN 아님)
3. `functional_map.png` 생성
4. `k=0`→`k=80` 구간에서 `malicious_token_prob`이 유의미하게 변화 (knockout이 실제로 적용됨)
5. `k=0`에서 `malicious_token_prob`, `read_token_prob` 둘 다 0 아님

결과 기록: `results/2026-07-27_colab_smoketest/README.md`

## 2. Phase 1~3 — 본 실험 (1.5B), VRAM 실측

먼저 6개 템플릿(도메인 6개 전부, 스타일 1종)으로 끝까지 도는지 확인.

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --family qwen2 \
  --device cuda \
  --topk 20 \
  --dataset_limit 6

문제없으면 전체 30개 템플릿으로 올린다 (`--dataset_limit` 제거).

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --family qwen2 \
  --device cuda \
  --topk 20

여유가 있으면 3B로 올려서 같은 경향이 재현되는지 확인.

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --family qwen2 \
  --device cuda \
  --topk 20

**결과 해석 지표** (RUN.md, results/2026-07-27_colab_phase1to3/README.md 참고):
- `jaccard(read, internal)`, `jaccard(read, external)` 낮음 → read와 instruction-following이 분리됨
- `jaccard(internal, external)` 높음 → 내부 응답 오염과 tool-call 오염이 같은 control head 회로를 공유
  (idea1의 "control head 한 번 찾아 knockout하면 둘 다 막힘" 전제를 지지)
- `control_heads_both` → Phase 1 knockout 후보 목록

**실측 결과 요약 (2026-07-27, Colab T4)**

| 모델 | 템플릿 수 | jaccard(read,internal) | jaccard(read,external) | jaccard(internal,external) | k=0 malicious | k=0 read |
|---|---|---|---|---|---|---|
| 0.5B | 2 (smoke) | 0.143 | 0.250 | 0.481 | 0.9386 | 0.6071 |
| 1.5B | 6  | 0.290 | 0.290 | 0.667 | 0.9914 | 0.4611 |
| 1.5B | 30 | 0.290 | 0.290 | 0.538 | 0.9094 | 0.4371 |
| 3B   | 30 | 0.333 | 0.290 | 0.538 | 0.9998 | 0.0067 ⚠️ |

⚠️ 3B의 `read_token_prob=0.0067`은 knockout 효과가 아니라 **측정 버그**였음 — 아래 3번 섹션 참고.

## 3. 디버깅 — 3B에서 `read_token_prob`이 비정상적으로 낮게 나온 원인

`debug_read_target.py`로 `read_clean`/`read_injected` 프롬프트에서 모델이 실제로 어떤 토큰을
이어 쓰는지 직접 확인한다 (lxt monkey-patch 없이 순수 HF만 사용).

In [ ]:
%cd /content/head_poc
!python debug_read_target.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --family qwen2 \
  --device cuda \
  --dataset_limit 3

**발견한 원인**: `READ_PREFIX="The answer is"` 뒤에 3B 모델은 거의 항상(77~95% 확률) `" that"`을
먼저 쓰고(`"The answer is that ~~~ 3pm"`), 우리가 재던 즉시-다음-토큰 자리(`read_target`, 예: `" 3pm"`)는
노이즈 수준의 확률(0.00001~0.02)만 가짐. `greedy continuation`을 보면 모델은 실제로는 정답을 정확히
말하고 있었음 — read 실패가 아니라 **측정 지점이 3B의 실제 응답 스타일과 안 맞은 것**.

**적용한 수정** (`dataset.py`, 커밋 `bc0acdc`):
- `READ_PREFIX`: `"The answer is"` → `"Answer:"`
- `READ_SYSTEM_SUFFIX` 추가: `"Respond with only the requested value, no extra words."`
  (read 계열 모드에서만 system prompt에 붙음, internal/external엔 영향 없음)

저장소에 이미 반영/커밋되어 있으므로, 아래 셀로 최신 코드를 받아서 다시 검증한다.

In [ ]:
%cd /content/head_poc
!git pull

In [ ]:
%cd /content/head_poc
!python debug_read_target.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --family qwen2 \
  --device cuda \
  --dataset_limit 3
# 확인할 것: read_target(예: " 3pm")이 top-1~2 근처로 올라오는지,
# greedy continuation이 "Answer: 3pm"처럼 바로 값이 나오는지

수정이 확인되면, 2번 섹션의 1.5B/3B 본 실험을 다시 돌려서 `read_token_prob` 수치를 갱신한다.

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --family qwen2 \
  --device cuda \
  --topk 20

## 4. (선택) Phase 4 — 7B급 스케일 검증

`--four_bit` 필요. `attn_relevance.py`는 gradient checkpointing과 상극이므로, head relevance
자체보다는 1.5B/3B에서 찾은 control head가 7B에서도 knockout으로 먹히는지(edge_ablation.py,
forward-only라 4bit와 무관하게 항상 안전) 검증하는 걸 우선한다. RUN.md 5단계 참고.

In [ ]:
%cd /content/head_poc
!python run_pipeline.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --family qwen2 \
  --device cuda \
  --four_bit \
  --topk 20

## 5. 다음 할 일 (TODO.md 참고)

**채널 분기(internal vs external) 담당 head 분리 실험**: 지금까지는 `internal_heads ∩ external_heads`
(공통 control head)만 봤음. "명령을 따르기로 한 뒤 자유 텍스트로 낼지 tool-call로 낼지 가르는 head"는
`external_heads - internal_heads` / `internal_heads - external_heads` (대칭차)로 봐야 함 —
아직 `head_ranking.py`에 구현 안 됨. 자세한 내용은 프로젝트 루트 `TODO.md` 참고.